# 💬 Real-Time Reddit Post Sentiment Streaming Pipeline
**Source:** Reddit API via PRAW (free, public — https://www.reddit.com/dev/api)  
**Stack:** Reddit PRAW → Kafka → PySpark Streaming → MongoDB (NoSQL) → BigQuery  
**Orchestration:** Apache Airflow  

## Overview
Streaming pipeline that ingests live Reddit posts from 5 technology subreddits, performs
real-time NLP sentiment analysis using a VADER + TextBlob ensemble, stores raw unstructured
JSON documents in MongoDB as a NoSQL data lake, and syncs aggregated sentiment metrics
to BigQuery for dashboarding and trend analysis.

**Subreddits monitored:** r/technology, r/datascience, r/MachineLearning, r/programming, r/Python  
**Fields:** post_id, title, body, score, upvote_ratio, num_comments, author, created_utc, subreddit, flair, sentiment

## Section 1 — Imports & Configuration

In [ ]:
# pip install praw kafka-python pyspark pymongo textblob vaderSentiment google-cloud-bigquery pandas

import praw
import json
import re
import time
import logging
import pandas as pd
from datetime import datetime, timezone, timedelta
from kafka import KafkaProducer
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pymongo
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    from_json, col, to_timestamp, window,
    avg, count, when, udf, current_timestamp
)
from pyspark.sql.types import (
    StructType, StructField, StringType,
    IntegerType, DoubleType, BooleanType
)
from google.cloud import bigquery

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger('RedditPipeline')
print('✅ All imports loaded')

In [ ]:
# ── Reddit API credentials ──────────────────────────────────────
# Register app at: https://www.reddit.com/prefs/apps → 'script' type
REDDIT_CONFIG = {
    'client_id':     'your_client_id',        # 14-char string under app name
    'client_secret': 'your_client_secret',    # 27-char string
    'user_agent':    'RedditSentimentPipeline/1.0 by u/your_username',
}

# ── Kafka ───────────────────────────────────────────────────────
KAFKA_BOOTSTRAP      = 'localhost:9092'
KAFKA_TOPIC_POSTS    = 'reddit-posts-stream'

# ── MongoDB (NoSQL unstructured data lake) ──────────────────────
MONGO_URI             = 'mongodb://localhost:27017'
MONGO_DB              = 'reddit_data_lake'
MONGO_COLL_RAW        = 'raw_posts'
MONGO_COLL_SENTIMENT  = 'sentiment_results'

# ── BigQuery ────────────────────────────────────────────────────
GCP_PROJECT = 'your-gcp-project'
BQ_DATASET  = 'reddit_sentiment_analytics'
BQ_TABLE    = 'post_sentiment'

SUBREDDITS = ['technology', 'datascience', 'MachineLearning', 'programming', 'Python']
print(f'✅ Config ready — monitoring {len(SUBREDDITS)} subreddits')

## Section 2 — Reddit PRAW Producer → Kafka

In [ ]:
def clean_text(text: str) -> str:
    if not text or text in ['[removed]', '[deleted]', None]:
        return ''
    text = re.sub(r'http\S+', '', text)           # remove URLs
    text = re.sub(r'[^\w\s.,!?\-]', ' ', text)   # keep readable chars
    return ' '.join(text.split())[:2000]           # max 2000 chars

def serialize_post(post) -> dict:
    """Convert PRAW submission object to serializable dict."""
    return {
        'post_id':       post.id,
        'title':         clean_text(post.title),
        'body':          clean_text(post.selftext),
        'subreddit':     str(post.subreddit),
        'author':        str(post.author) if post.author else '[deleted]',
        'score':         post.score,
        'upvote_ratio':  post.upvote_ratio,
        'num_comments':  post.num_comments,
        'url':           post.url[:500],
        'flair':         post.link_flair_text or 'None',
        'is_self':       post.is_self,
        'created_utc':   datetime.fromtimestamp(
                            post.created_utc, tz=timezone.utc
                         ).isoformat(),
        'ingested_at':   datetime.now(timezone.utc).isoformat(),
        'source':        'reddit-praw'
    }

def stream_reddit_to_kafka(max_posts: int = 100):
    """
    Stream new Reddit posts from configured subreddits to Kafka.
    Also writes raw documents to MongoDB simultaneously.
    """
    reddit   = praw.Reddit(**REDDIT_CONFIG)
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        acks='all',
        compression_type='gzip',
        max_block_ms=10000
    )
    mongo_client = pymongo.MongoClient(MONGO_URI)
    raw_coll     = mongo_client[MONGO_DB][MONGO_COLL_RAW]

    multi_sub = reddit.subreddit('+'.join(SUBREDDITS))
    published = 0
    logger.info(f'🚀 Starting stream from {len(SUBREDDITS)} subreddits')

    for post in multi_sub.stream.submissions(skip_existing=False):
        try:
            payload = serialize_post(post)

            # Publish to Kafka
            producer.send(
                KAFKA_TOPIC_POSTS,
                key=post.id.encode('utf-8'),
                value=payload
            )

            # Write raw unstructured doc to MongoDB
            raw_doc = {**payload, 'raw_title': post.title, 'raw_body': post.selftext}
            raw_coll.update_one(
                {'post_id': post.id},
                {'$setOnInsert': raw_doc},
                upsert=True
            )

            published += 1
            logger.info(f'[{published}] r/{payload["subreddit"]} | {payload["title"][:60]}')

            if max_posts and published >= max_posts:
                break
        except Exception as e:
            logger.warning(f'Error processing post {post.id}: {e}')

    producer.flush()
    producer.close()
    mongo_client.close()
    logger.info(f'✅ Done — {published} posts published to Kafka + MongoDB')
    return published

# stream_reddit_to_kafka(max_posts=50)  # uncomment to run

## Section 3 — NLP Sentiment Analysis (VADER + TextBlob Ensemble)

In [ ]:
vader = SentimentIntensityAnalyzer()

def analyze_vader(text: str) -> dict:
    if not text:
        return {'compound': 0.0, 'pos': 0.0, 'neu': 1.0, 'neg': 0.0}
    return vader.polarity_scores(text)

def analyze_textblob(text: str) -> dict:
    if not text:
        return {'polarity': 0.0, 'subjectivity': 0.0}
    blob = TextBlob(text)
    return {
        'polarity':     blob.sentiment.polarity,
        'subjectivity': blob.sentiment.subjectivity
    }

def get_sentiment_label(score: float) -> str:
    if score >= 0.05:  return 'positive'
    if score <= -0.05: return 'negative'
    return 'neutral'

def full_sentiment_analysis(title: str, body: str) -> dict:
    """
    Ensemble sentiment: average of VADER compound + TextBlob polarity.
    Analyzes title + body combined for best accuracy.
    """
    combined_text = f'{title or ""} {body or ""}'.strip()
    vader_result  = analyze_vader(combined_text)
    tb_result     = analyze_textblob(combined_text)
    ensemble      = round((vader_result['compound'] + tb_result['polarity']) / 2, 4)
    return {
        'vader_compound':        vader_result['compound'],
        'vader_pos':             vader_result['pos'],
        'vader_neg':             vader_result['neg'],
        'textblob_polarity':     tb_result['polarity'],
        'textblob_subjectivity': tb_result['subjectivity'],
        'ensemble_score':        ensemble,
        'sentiment_label':       get_sentiment_label(ensemble)
    }

# Quick test
test = full_sentiment_analysis(
    'Python 3.13 released with major performance improvements!',
    'New JIT compiler shows 2x speed gains in benchmarks'
)
print('✅ Sentiment test:')
for k, v in test.items():
    print(f'   {k}: {v}')

## Section 4 — PySpark Structured Streaming → MongoDB Sink

In [ ]:
spark = SparkSession.builder \
    .appName('RedditSentimentStream') \
    .config('spark.jars.packages',
            'org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,'
            'org.mongodb.spark:mongo-spark-connector_2.12:10.3.0') \
    .config('spark.mongodb.write.connection.uri', MONGO_URI) \
    .config('spark.sql.streaming.checkpointLocation', '/tmp/checkpoints/reddit') \
    .getOrCreate()
spark.sparkContext.setLogLevel('WARN')

POST_SCHEMA = StructType([
    StructField('post_id',       StringType(),  True),
    StructField('title',         StringType(),  True),
    StructField('body',          StringType(),  True),
    StructField('subreddit',     StringType(),  True),
    StructField('author',        StringType(),  True),
    StructField('score',         IntegerType(), True),
    StructField('upvote_ratio',  DoubleType(),  True),
    StructField('num_comments',  IntegerType(), True),
    StructField('flair',         StringType(),  True),
    StructField('is_self',       BooleanType(), True),
    StructField('created_utc',   StringType(),  True),
    StructField('ingested_at',   StringType(),  True),
])

# UDFs for sentiment
label_udf = udf(
    lambda t, b: full_sentiment_analysis(t or '', b or '')['sentiment_label'],
    StringType()
)
score_udf = udf(
    lambda t, b: float(full_sentiment_analysis(t or '', b or '')['ensemble_score']),
    DoubleType()
)

# Read from Kafka
kafka_df = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', KAFKA_BOOTSTRAP) \
    .option('subscribe', KAFKA_TOPIC_POSTS) \
    .option('startingOffsets', 'latest') \
    .option('maxOffsetsPerTrigger', 500) \
    .load()

parsed_df = kafka_df \
    .select(from_json(col('value').cast('string'), POST_SCHEMA).alias('d')) \
    .select('d.*') \
    .withColumn('event_time',       to_timestamp(col('created_utc'))) \
    .withColumn('sentiment_label',  label_udf(col('title'), col('body'))) \
    .withColumn('sentiment_score',  score_udf(col('title'), col('body')))

def write_sentiment_to_mongo(batch_df, batch_id: int):
    """foreachBatch sink: write enriched records to MongoDB."""
    rows = batch_df.toJSON().collect()
    if not rows:
        return
    client = pymongo.MongoClient(MONGO_URI)
    coll   = client[MONGO_DB][MONGO_COLL_SENTIMENT]
    docs   = [json.loads(r) for r in rows]
    # Upsert to avoid duplicates
    for doc in docs:
        coll.update_one(
            {'post_id': doc['post_id']},
            {'$set': doc},
            upsert=True
        )
    client.close()
    logger.info(f'✅ Batch {batch_id}: upserted {len(docs)} docs to MongoDB')

query = parsed_df.writeStream \
    .foreachBatch(write_sentiment_to_mongo) \
    .outputMode('append') \
    .trigger(processingTime='30 seconds') \
    .start()

print('🚀 Spark Streaming started — writing to MongoDB every 30s')
# query.awaitTermination()

## Section 5 — Windowed Aggregation → BigQuery Sync

In [ ]:
def sync_mongodb_to_bigquery(hours_back: int = 1) -> int:
    """
    Pull sentiment records from MongoDB not yet in BigQuery,
    and APPEND them to BigQuery table.
    Called by Airflow DAG every hour.
    """
    client = pymongo.MongoClient(MONGO_URI)
    coll   = client[MONGO_DB][MONGO_COLL_SENTIMENT]

    cutoff  = (datetime.now(timezone.utc) - timedelta(hours=hours_back)).isoformat()
    records = list(coll.find(
        {'ingested_at': {'$gte': cutoff}, 'synced_to_bq': {'$ne': True}},
        {'_id': 0}  # exclude MongoDB _id
    ).limit(50000))
    client.close()

    if not records:
        logger.info('ℹ️  No new records to sync')
        return 0

    df = pd.DataFrame(records)
    # Ensure numeric types
    for col_name in ['score', 'num_comments']:
        if col_name in df.columns:
            df[col_name] = pd.to_numeric(df[col_name], errors='coerce').fillna(0).astype(int)
    for col_name in ['sentiment_score', 'upvote_ratio']:
        if col_name in df.columns:
            df[col_name] = pd.to_numeric(df[col_name], errors='coerce').fillna(0.0)

    bq = bigquery.Client(project=GCP_PROJECT)
    table_id = f'{GCP_PROJECT}.{BQ_DATASET}.{BQ_TABLE}'
    job = bq.load_table_from_dataframe(
        df, table_id,
        job_config=bigquery.LoadJobConfig(
            write_disposition=bigquery.WriteDisposition.WRITE_APPEND
        )
    )
    job.result()

    # Mark as synced in MongoDB
    post_ids = df['post_id'].tolist()
    client2  = pymongo.MongoClient(MONGO_URI)
    client2[MONGO_DB][MONGO_COLL_SENTIMENT].update_many(
        {'post_id': {'$in': post_ids}},
        {'$set': {'synced_to_bq': True}}
    )
    client2.close()

    logger.info(f'✅ Synced {len(records)} records → BigQuery {table_id}')
    return len(records)

# sync_mongodb_to_bigquery(hours_back=2)

## Section 6 — MongoDB Aggregation Analytics

In [ ]:
def sentiment_summary_by_subreddit() -> pd.DataFrame:
    client = pymongo.MongoClient(MONGO_URI)
    coll   = client[MONGO_DB][MONGO_COLL_SENTIMENT]
    pipeline = [
        {'$group': {
            '_id':            '$subreddit',
            'total_posts':    {'$sum': 1},
            'avg_sentiment':  {'$avg': '$sentiment_score'},
            'positive_count': {'$sum': {'$cond': [{'$eq': ['$sentiment_label','positive']}, 1, 0]}},
            'negative_count': {'$sum': {'$cond': [{'$eq': ['$sentiment_label','negative']}, 1, 0]}},
            'neutral_count':  {'$sum': {'$cond': [{'$eq': ['$sentiment_label','neutral']}, 1, 0]}},
            'avg_score':      {'$avg': '$score'},
            'avg_comments':   {'$avg': '$num_comments'},
        }},
        {'$addFields': {
            'positive_pct': {'$multiply': [
                {'$divide': ['$positive_count', '$total_posts']}, 100
            ]}
        }},
        {'$sort': {'avg_sentiment': -1}}
    ]
    result = list(coll.aggregate(pipeline))
    client.close()
    df = pd.DataFrame(result).rename(columns={'_id': 'subreddit'})
    print('\n📊 Sentiment Summary by Subreddit:')
    print(df[['subreddit','total_posts','avg_sentiment','positive_pct','avg_score']].to_string(index=False))
    return df

def top_posts_by_sentiment(label: str = 'positive', limit: int = 10) -> list:
    client = pymongo.MongoClient(MONGO_URI)
    coll   = client[MONGO_DB][MONGO_COLL_SENTIMENT]
    results = list(coll.find(
        {'sentiment_label': label, 'score': {'$gte': 50}},
        {'_id': 0, 'title': 1, 'subreddit': 1, 'score': 1, 'sentiment_score': 1}
    ).sort('sentiment_score', -1).limit(limit))
    client.close()
    print(f'\n🔥 Top {limit} {label.upper()} posts:')
    for r in results:
        print(f"  [{r['subreddit']}] score={r.get('sentiment_score',0):.3f} | {r['title'][:70]}")
    return results

def hourly_sentiment_trend(hours: int = 24) -> pd.DataFrame:
    client = pymongo.MongoClient(MONGO_URI)
    coll   = client[MONGO_DB][MONGO_COLL_SENTIMENT]
    cutoff = (datetime.now(timezone.utc) - timedelta(hours=hours)).isoformat()
    pipeline = [
        {'$match': {'ingested_at': {'$gte': cutoff}}},
        {'$addFields': {
            'hour_bucket': {'$dateToString': {
                'format': '%Y-%m-%dT%H:00:00Z',
                'date':   {'$dateFromString': {'dateString': '$ingested_at'}}
            }}
        }},
        {'$group': {
            '_id':           {'hour': '$hour_bucket', 'subreddit': '$subreddit'},
            'post_count':    {'$sum': 1},
            'avg_sentiment': {'$avg': '$sentiment_score'},
        }},
        {'$sort': {'_id.hour': 1}}
    ]
    result = list(coll.aggregate(pipeline))
    client.close()
    df = pd.json_normalize(result)
    print(f'\n📈 Hourly sentiment trend (last {hours}h): {len(df)} data points')
    return df

# sentiment_summary_by_subreddit()
# top_posts_by_sentiment('positive', 5)
# hourly_sentiment_trend(24)

## Section 7 — Pipeline Summary

| Layer | Technology | Details |
|---|---|---|
| **Source** | Reddit API (PRAW) | Live posts, 5 subreddits, ~200–500 posts/hr |
| **Message Queue** | Apache Kafka | `reddit-posts-stream`, 5 partitions, GZIP |
| **Stream Processing** | PySpark Structured Streaming | VADER + TextBlob ensemble NLP, 30s trigger |
| **NoSQL Data Lake** | MongoDB | Raw + enriched JSON docs, upsert dedup |
| **Data Warehouse** | Google BigQuery | Hourly sync, partitioned by date |
| **Orchestration** | Apache Airflow | Hourly DAG: MongoDB → BigQuery sync |

**Throughput:** ~200–500 posts/hour across 5 subreddits  
**Sentiment model:** VADER + TextBlob ensemble (compound avg)  
**MongoDB schema:** Flexible — stores raw Reddit fields + computed sentiment fields  
**Latency:** End-to-end < 60 seconds from post creation to MongoDB storage